In [ ]:
# %% [markdown]
# # 📱 Multi-App Google Play Review Fetcher (Project 1 - Phase 1)
# Fetches real customer feedback across 5 popular apps, maps ratings to sentiment, and exports a clean CSV.

# %% [code]
# Install dependencies (safe to run multiple times)
!pip install -q google-play-scraper pandas tqdm

from google_play_scraper import reviews, Sort
import pandas as pd
from tqdm import tqdm
import time
import random

# 🔧 Configuration
APP_LIST = [
    {"id": "com.jumia.android", "name": "Jumia (E-commerce)"},
    {"id": "com.glovo", "name": "Glovo (Food Delivery)"},
    {"id": "com.ubercab", "name": "Uber (Ride-hailing)"},
    {"id": "com.booking", "name": "Booking.com (Travel)"},
    {"id": "com.amazon.mShop.android.shopping", "name": "Amazon (Retail)"}
]

REVIEWS_PER_APP = 500  # ~1,500 total reviews (balanced, rate-limit friendly)
LANG = "en"
COUNTRY = "us"  # Change to "tn" for Tunisia, but "us" yields more English text
SORT = Sort.MOST_RELEVANT  # or Sort.NEWEST for recent feedback

all_reviews = []

print("🔍 Fetching real customer reviews across 5 apps...\n")
for app in tqdm(APP_LIST, desc="Apps", unit="app"):
    try:
        result, _ = reviews(
            app["id"], 
            lang=LANG, 
            country=COUNTRY, 
            sort=SORT, 
            count=REVIEWS_PER_APP
        )
        for r in result:
            r["app_name"] = app["name"]
            r["app_id"] = app["id"]
        all_reviews.extend(result)
        print(f"✅ {app['name']}: {len(result)} reviews fetched")
        time.sleep(random.uniform(1.5, 3.0))  # Respectful delay to avoid IP throttling
    except Exception as e:
        print(f"⚠️ Failed to fetch {app['name']}: {e}")

# Convert to DataFrame
df = pd.DataFrame(all_reviews)
if df.empty:
    raise RuntimeError("No reviews fetched. Check app IDs, network, or library version.")

# 🏷️ Automatic Sentiment Labeling from Ratings
def label_sentiment(score):
    if score >= 4.0: return "positive"
    elif score <= 2.0: return "negative"
    else: return "neutral"

df["sentiment"] = df["score"].apply(label_sentiment)

# 🧼 Basic Cleaning & Column Selection
df_clean = df[["app_name", "userName", "content", "score", "sentiment", "at"]].copy()
df_clean = df_clean.rename(columns={"content": "text", "at": "date"})
df_clean = df_clean.dropna(subset=["text"])
df_clean = df_clean[df_clean["text"].str.strip() != ""].reset_index(drop=True)

# 💾 Export
import os
os.makedirs("data", exist_ok=True)
df_clean.to_csv("data/gplay_multi_app_reviews.csv", index=False)

print(f"\n✅ Saved {len(df_clean)} reviews to data/gplay_multi_app_reviews.csv")
print("📊 Sentiment Distribution:")
print(df_clean["sentiment"].value_counts())
print("\n📈 App-wise Distribution:")
print(df_clean["app_name"].value_counts())

In [ ]:
import pandas as pd
import os

DATA_DIR = "data"

# Load ONLY 5,000 from Yelp (fast, sufficient for DistilBERT)
yelp_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), 
                      header=None, 
                      names=["label", "text"], 
                      nrows=5000)  # ← This limits to 5k

yelp_df["label"] = yelp_df["label"].map({1: "negative", 2: "positive"})
yelp_df["source"] = "yelp_public"

# Load your Google Play data (2,500 reviews)
gplay_df = pd.read_csv(os.path.join(DATA_DIR, "gplay_multi_app_reviews.csv"))
gplay_df = gplay_df.rename(columns={"sentiment": "label", "content": "text"})
gplay_df["source"] = "google_play"

# Merge
df = pd.concat([yelp_df, gplay_df], ignore_index=True)
df = df.rename(columns={"label": "sentiment", "text": "review"})

print(f"Total: {len(df)} reviews")
print(df["sentiment"].value_counts())
print(f"Sources: {df['source'].value_counts().to_dict()}")
